### combine collar & eartag data with collocated SIC, SIT, SD and SR

In [ ]:
import glob
import pandas as pd
import numpy as np
import datetime
from pyproj import Proj, Transformer
from scipy.spatial.kdtree import KDTree
import math
from netCDF4 import Dataset
import matplotlib.pyplot as plt
import os
import itertools

### combine GPS files

In [ ]:
collars_df = []

### collar data
for year in np.arange(2010,2023):
    datapath = f'/Users/carmennab/Dropbox/pb_new/data/bears/collars/{year}/'
    files = glob.glob(f'{datapath}*.csv')
    for file in files:
        data = pd.read_csv(file)
        ID = data['Xnumber']
        date_time = data['loc_date_time']
#         date = [datetime.datetime.strptime(dt, '%Y.%m.%d %H:%M:%S').date() for dt in date_time]
        long = data['longitude']
        lat = data['latitude']
        mort = data['mortality']
        time = data['loc_time']

        for i, d, lo, la, m, t in zip(ID,date_time,long,lat,mort,time):
            collars_df.append({'ID':i,'datetime':d,'longitude':lo,'latitude':la,'mortality':m,'time':t,'type':'collar'})

collars_df = pd.DataFrame(collars_df).drop_duplicates().dropna()
dates = [datetime.datetime.strptime(dt, '%Y.%m.%d %H:%M:%S').date() for dt in collars_df['datetime']]
collars_df['date'] = dates

collars_df = collars_df[collars_df['ID'] != 'Unknown'] ### remove data points where no bear ID 
collars_df = collars_df[collars_df['mortality'] != 'Yes'] ### remove data points where GPS has stopped
collars_df = collars_df.drop(columns=['mortality','datetime'])
collars_df

In [ ]:
eartags_df = []
    
### eartag data
datapath = f'/Users/carmennab/Dropbox/pb_new/data/bears/eartags/'
files = glob.glob(f'{datapath}*.csv')
for file in files:
    data = pd.read_csv(file)
    ID = data['Xnumber']
    date_time = data['loc_date_time']
    long = data['longitude']
    lat = data['latitude']
    mort = data['mortality']

    for i, d, lo, la, m, in zip(ID,date_time,long,lat,mort):
        eartags_df.append({'ID':i,'datetime':d,'longitude':lo,'latitude':la,'mortality':m,'type':'eartag'})
          
eartags_df = pd.DataFrame(eartags_df).drop_duplicates().dropna()
dates = [datetime.datetime.strptime(dt, '%Y.%m.%d %H:%M:%S').date() for dt in eartags_df['datetime']]
eartags_df['date'] = dates

eartags_df = eartags_df[eartags_df['ID'] != 'Unknown'] ### remove data points where no bear ID 
eartags_df = eartags_df[eartags_df['mortality'] != 'Yes'] ### remove data points where GPS has stopped
eartags_df = eartags_df.drop(columns=['mortality','datetime'])
eartags_df

In [ ]:
combined_df = pd.concat([collars_df, eartags_df], ignore_index=True)

In [ ]:
combined_df

### add sex and age

In [ ]:
info_datapath = '/Users/carmennab/Dropbox/pb_new/data/bears/collar_info.csv'

info = pd.read_csv(info_datapath,usecols=['Bear','Sex','Age']).dropna()
info

In [ ]:
sex = [] ; age = []

for ID in combined_df['ID']:
    try:
        a = info[info['Bear']==ID]['Age'].iloc[0]
    except:
        a = np.nan
    
    try:
        s = info[info['Bear']==ID]['Sex'].iloc[0]
    except: s = np.nan
    
    sex.append(s) ; age.append(a)

combined_df['sex'] = sex ; combined_df['age'] = age

### add EASE locations 

In [ ]:
def lonlat_to_xy(coords_1, coords_2, hemisphere, inverse=False):

    """Converts between longitude/latitude and EASE xy coordinates.
 
    Args:
        lon (float): WGS84 longitude
        lat (float): WGS84 latitude
        hemisphere (string): 'n' or 's'
        inverse (bool): if true, converts xy to lon/lat
 
    Returns:
        tuple: pair of xy or lon/lat values
    """

    EASE_Proj = {'n': 'EPSG:3408',
                 's': 'EPGS:3409'}
    
    WGS_Proj = 'EPSG:4326'
    
    for coords in [coords_1, coords_2]: assert isinstance(coords,(np.ndarray,list))

    if inverse == False: # lonlat to xy
        
        lon, lat = coords_1, coords_2
        
        transformer = Transformer.from_crs(WGS_Proj, EASE_Proj[hemisphere])
        
        x, y = transformer.transform(lat, lon)
        
        return (x, y)

    else: # xy to lonlat
        
        x, y = coords_1, coords_2
        
        transformer = Transformer.from_crs(EASE_Proj[hemisphere], WGS_Proj)
        
        lat, lon = transformer.transform(x, y)
        
        return (lon, lat)

In [ ]:
datapath_aux = '/Users/carmennab/Dropbox/pb_new/auxiliary/'
ease_lons = np.load(f'{datapath_aux}lon_cent.npy')
ease_lats = np.load(f'{datapath_aux}lat_cent.npy')

# convert into x,y:
ease_x, ease_y = lonlat_to_xy(ease_lons,ease_lats,hemisphere='n')

# create KDTree to find EASE grid cell closest to buoy:
tree = KDTree(list(zip(ease_x.ravel(),ease_y.ravel())))

In [ ]:
bear_x,bear_y = lonlat_to_xy(np.array(combined_df['longitude']),np.array(combined_df['latitude']),hemisphere='n')
combined_df['x'] = bear_x
combined_df['y'] = bear_y

In [ ]:
ind2ds = [np.unravel_index(tree.query([x,y])[1], (ease_lats.shape[0],ease_lats.shape[1])) for x,y in zip(bear_x,bear_y)]
combined_df['ind2d'] = ind2ds

In [ ]:
combined_df = combined_df.dropna()

### add collocated ice and snow

In [ ]:
### SD and SIT
datapath_sit = '/Users/carmennab/Dropbox/pb_new/data/SIT/CS2SMOS/'
SD = [] ; SIT = [] 

for date, ind2d in zip(combined_df['date'],combined_df['ind2d']):
    str_date = date.strftime('%Y%m%d')
    filepath_sit = f'{datapath_sit}{date.year}/{str_date}.nc'
    if os.path.exists(filepath_sit) == True: 
        data = Dataset(filepath_sit)
        SD.append(np.array(data['snow_depth'])[ind2d])
        SIT.append(np.array(data['sea_ice_thickness'])[ind2d])
     
    else:
        SIT.append(np.nan) 
        SD.append(np.nan) 
        
combined_df['SD'] = SD
combined_df['SIT'] = SIT

In [ ]:
### SR
datapath_sit = '/Users/carmennab/Dropbox/pb_new/daily_mean_roughness/25km_ease/'
SR = []  

for date, ind2d in zip(combined_df['date'],combined_df['ind2d']):
    str_date = date.strftime('%Y-%m-%d')
    filepath_sit = f'{datapath_sit}{date.year}/{str_date}.nc'
    if os.path.exists(filepath_sit) == True: 
        data = Dataset(filepath_sit)
        SR.append(np.array(data['roughness'])[ind2d])
    else:
        SR.append(np.nan) 
        
combined_df['SR'] = SR

In [ ]:
region_mask = np.array(Dataset(f'{datapath_aux}NSIDC_Regions_Mask_ease.nc')['mask'])
region_mask[ease_lats<55] = 0
region_mask[ease_lats>62] = 0
region_mask[ease_lons>-76] = 0

In [ ]:
### SIC
datapath_sic = '/Users/carmennab/Dropbox/pb_new/data/SIC/'
SIC = []  

for date, ind2d in zip(combined_df['date'],combined_df['ind2d']):
    str_date = date.strftime('%Y%m%d')
    filepath_sic = f'{datapath_sic}{str_date[0:4]}/{str_date}.nc'
    if os.path.exists(filepath_sic) == True: 
        data = Dataset(filepath_sic)
        sic = np.array(data['sic'])
        sic[~np.isin(region_mask,4)] = np.nan
        SIC.append(sic[ind2d])
    else:
        SIC.append(np.nan) 
        
combined_df['SIC'] = SIC

### calculate distance to land

In [ ]:
region_mask = np.array(Dataset(f'{datapath_aux}NSIDC_Regions_Mask_ease.nc')['mask'])
ease_xs, ease_ys = lonlat_to_xy(ease_lons,ease_lats,hemisphere='n')
land_xs = ease_xs[~np.isin(region_mask,4)] ; land_ys = ease_ys[~np.isin(region_mask,4)]
min_ds = np.full(ease_xs.shape,np.nan)

for i, j in itertools.product(np.arange(ease_xs.shape[0]),np.arange(ease_xs.shape[1])):
    ease_x = ease_xs[i,j] ; ease_y = ease_ys[i,j]
    d_to_land = [math.sqrt((land_x-ease_x)**2 + (land_y-ease_y)**2) for land_x,land_y in zip(land_xs,land_ys)]
    min_ds[i,j] = np.nanmin(d_to_land)

In [ ]:
d_to_land = []  
for ind2d in combined_df['ind2d']:
    d_to_land.append(min_ds[ind2d])
    
combined_df['d_to_land'] = d_to_land

### save combined df

In [ ]:
combined_df

In [ ]:
combined_df.to_csv('/Users/carmennab/Dropbox/pb_new/data/bears/data_combined.csv',index=False)

In [ ]:
### calculate number of individuals in each class

for age in ['Adult','Subadult']:
    for sex in ['F','M']:
        data = combined_df[(combined_df['sex']==sex) & (combined_df['age']==age)]
        num = len(set(data['ID']))
    
        print(f'{sex} {age}: {num}')
